# IEEE-CIS Fraud Detection — ML Experiment Pipeline
**Kaggle Notebook | T4 GPU | Full Dataset (590k) | NeighborLoader GNNs**

Baselines: LR (Class Weighted) · LR (SMOTE) · RF (Class Weighted) · RF (SMOTE) · LightGBM · Isolation Forest

GNNs: GCN · GAT · GraphSAGE · CS-GraphSAGE

> Session options → Accelerator → GPU T4 x2

> Run cells top to bottom. **Restart kernel before running** to ensure clean imports.

## Step 1: Install PyTorch Geometric

In [1]:
import subprocess, sys

torch_ver = '2.10.0'
cuda_ver  = '128'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html'

# Install PyG, torch-sparse (enables NeighborLoader), torch-scatter
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch-geometric', 'torch-sparse', 'torch-scatter',
                '-f', wheel_url], check=True)

# Install imbalanced-learn for SMOTE
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'imbalanced-learn'], check=True)

import torch, torch_geometric, torch_sparse
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'PyG         : {torch_geometric.__version__}')
print(f'torch-sparse: {torch_sparse.__version__}')
print('Ready.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 49.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.9 MB/s eta 0:00:00
PyTorch     : 2.10.0+cu128
CUDA        : True
GPU         : Tesla T4
VRAM        : 15.6 GB
PyG         : 2.7.0
torch-sparse: 0.6.18+pt210cu128
Ready.


## Step 2: Load Data

In [2]:
import pandas as pd
import numpy as np
import gc, time, warnings
warnings.filterwarnings('ignore')

DATA_PATH = '/kaggle/input/competitions/ieee-fraud-detection/'

def reduce_mem(df):
    """Cast columns to smallest viable dtype — halves RAM usage."""
    for col in df.columns:
        ct = df[col].dtype
        if ct == object: continue
        c_min, c_max = df[col].min(), df[col].max()
        if str(ct)[:3] == 'int':
            if   c_min > np.iinfo(np.int8).min  and c_max < np.iinfo(np.int8).max:  df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
        else:
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
    return df

print('Loading...')
train_txn = reduce_mem(pd.read_csv(DATA_PATH + 'train_transaction.csv'))
train_id  = reduce_mem(pd.read_csv(DATA_PATH + 'train_identity.csv'))
train     = train_txn.merge(train_id, on='TransactionID', how='left')
del train_txn, train_id; gc.collect()

print(f'Shape      : {train.shape}')
print(f'Fraud rate : {train["isFraud"].mean():.4f}  ({int(train["isFraud"].sum()):,} fraud)')
print(f'RAM        : {train.memory_usage().sum()/1e6:.0f} MB')


Loading...
Shape      : (590540, 434)
Fraud rate : 0.0350  (20,663 fraud)
RAM        : 1095 MB


## Step 3: Preprocessing

In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

y_all = train['isFraud'].values.astype(np.int8)
X_all = train.drop(['isFraud','TransactionID'], axis=1).copy()

cat_cols = X_all.select_dtypes(include='object').columns.tolist()
print(f'Encoding {len(cat_cols)} categorical columns...')

for col in cat_cols:
    le = LabelEncoder()
    X_all[col] = X_all[col].fillna('__NA__')
    X_all[col] = le.fit_transform(X_all[col].astype(str)).astype(np.int16)

X_all = X_all.fillna(-999).astype(np.float32)
print(f'Features   : {X_all.shape}  RAM: {X_all.memory_usage().sum()/1e6:.0f} MB')

X_temp, X_test, y_temp, y_test = train_test_split(
    X_all, y_all, test_size=0.15, random_state=42, stratify=y_all)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)
del X_temp, X_all; gc.collect()

print(f'Train: {X_train.shape}  fraud={y_train.mean():.4f}')
print(f'Val  : {X_val.shape}  fraud={y_val.mean():.4f}')
print(f'Test : {X_test.shape}  fraud={y_test.mean():.4f}')


Encoding 31 categorical columns...
Features   : (590540, 432)  RAM: 1020 MB
Train: (413614, 432)  fraud=0.0350
Val  : (88345, 432)  fraud=0.0350
Test : (88581, 432)  fraud=0.0350


## Step 4: Baseline Models

In [4]:
from sklearn.metrics import precision_score

results = {}

def evaluate(name, y_true, y_score, thresh=0.5):
    auc_roc   = roc_auc_score(y_true, y_score)
    auc_pr    = average_precision_score(y_true, y_score)
    preds     = (y_score >= thresh).astype(int)
    f1        = f1_score(y_true, preds, zero_division=0)
    precision = precision_score(y_true, preds, zero_division=0)
    k         = max(1, int(0.01 * len(y_true)))
    top_k     = np.argsort(y_score)[::-1][:k]
    r1pct     = float(y_true[top_k].mean())
    results[name] = dict(AUC_ROC=auc_roc, AUC_PR=auc_pr, F1=f1,
                         Precision=precision, Recall_at_1pct=r1pct)
    print(f'{name:<35} AUC-ROC={auc_roc:.4f}  AUC-PR={auc_pr:.4f}  '
          f'F1={f1:.4f}  Precision={precision:.4f}  '
          f'Recall@1%={r1pct:.4f}  (thresh={thresh:.2f})')


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score as f1_fn
from imblearn.over_sampling import SMOTE
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'imbalanced-learn'], check=True)

print('--- Logistic Regression (full training set) ---')
# LR is trained on the full training set.
# StandardScaler is applied — required for lbfgs convergence.
sc   = StandardScaler()
X_tr = sc.fit_transform(X_train.to_numpy(dtype=np.float32))
X_v  = sc.transform(X_val.to_numpy(dtype=np.float32))
X_te = sc.transform(X_test.to_numpy(dtype=np.float32))
print(f'  Training on {X_tr.shape[0]:,} samples  fraud={y_train.mean():.4f}')
t0   = time.time()

# Variant 1: Class weights
lr_cw = LogisticRegression(C=0.01, max_iter=200, class_weight='balanced',
                            solver='lbfgs', n_jobs=-1, random_state=42)
lr_cw.fit(X_tr, y_train)
p_cw  = lr_cw.predict_proba(X_te)[:,1]
vp_cw = lr_cw.predict_proba(X_v)[:,1]
best_f1, best_thresh = 0, 0.5
for thresh in np.arange(0.01, 0.99, 0.01):
    f1 = f1_fn(y_val, (vp_cw >= thresh).astype(int), zero_division=0)
    if f1 > best_f1: best_f1, best_thresh = f1, thresh
print(f'  [Class Weighted] threshold={best_thresh:.2f}  val F1={best_f1:.4f}')
evaluate('LR (Class Weighted)', y_test, p_cw, thresh=best_thresh)

# Variant 2: SMOTE
# SMOTE is memory-intensive on the full set — subsample to 100k for SMOTE
# then train LR on the resampled data. Evaluation is still on the full test set.
smote_sub = np.random.RandomState(42).choice(len(X_tr), 100_000, replace=False)
print(f'  Applying SMOTE to 100k subsample (memory-efficient)...')
smote = SMOTE(random_state=42, k_neighbors=5)
X_res, y_res = smote.fit_resample(X_tr[smote_sub], y_train[smote_sub])
print(f'  After SMOTE: {X_res.shape[0]:,} samples  fraud={y_res.mean():.4f}')
lr_sm = LogisticRegression(C=0.01, max_iter=200,
                            solver='lbfgs', n_jobs=-1, random_state=42)
lr_sm.fit(X_res, y_res)
p_sm  = lr_sm.predict_proba(X_te)[:,1]
vp_sm = lr_sm.predict_proba(X_v)[:,1]
best_f1, best_thresh = 0, 0.5
for thresh in np.arange(0.01, 0.99, 0.01):
    f1 = f1_fn(y_val, (vp_sm >= thresh).astype(int), zero_division=0)
    if f1 > best_f1: best_f1, best_thresh = f1, thresh
print(f'  [SMOTE] threshold={best_thresh:.2f}  val F1={best_f1:.4f}')
print(f'  Time: {time.time()-t0:.1f}s')
evaluate('LR (SMOTE)', y_test, p_sm, thresh=best_thresh)
del X_tr, X_res, X_v, X_te, lr_cw, lr_sm, sc
del p_cw, p_sm, vp_cw, vp_sm, y_res; gc.collect()


--- Logistic Regression (full training set) ---
  Training on 413,614 samples  fraud=0.0350


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  [Class Weighted] threshold=0.81  val F1=0.3306
LR (Class Weighted)                 AUC-ROC=0.8380  AUC-PR=0.2629  F1=0.3304  Precision=0.2972  Recall@1%=0.5141  (thresh=0.81)
  Applying SMOTE to 100k subsample (memory-efficient)...
  After SMOTE: 192,846 samples  fraud=0.5000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  [SMOTE] threshold=0.82  val F1=0.3246
  Time: 92.8s
LR (SMOTE)                          AUC-ROC=0.8302  AUC-PR=0.2457  F1=0.3207  Precision=0.2891  Recall@1%=0.4847  (thresh=0.82)


0

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score as f1_fn
from imblearn.over_sampling import SMOTE

print('--- Random Forest (full training set) ---')
print(f'  Training on {X_train.shape[0]:,} samples  fraud={y_train.mean():.4f}')
X_tr_np = X_train.to_numpy(dtype=np.float32)
X_te_np = X_test.to_numpy(dtype=np.float32)
X_v_np  = X_val.to_numpy(dtype=np.float32)
t0 = time.time()

# Variant 1: Class weights — full training set
rf_cw = RandomForestClassifier(n_estimators=200, max_depth=10,
                                class_weight='balanced',
                                n_jobs=-1, random_state=42)
rf_cw.fit(X_tr_np, y_train)
p_cw  = rf_cw.predict_proba(X_te_np)[:,1]
vp_cw = rf_cw.predict_proba(X_v_np)[:,1]
best_f1, best_thresh = 0, 0.5
for thresh in np.arange(0.01, 0.99, 0.01):
    f1 = f1_fn(y_val, (vp_cw >= thresh).astype(int), zero_division=0)
    if f1 > best_f1: best_f1, best_thresh = f1, thresh
print(f'  [Class Weighted] threshold={best_thresh:.2f}  val F1={best_f1:.4f}')
evaluate('RF (Class Weighted)', y_test, p_cw, thresh=best_thresh)
del rf_cw, p_cw, vp_cw; gc.collect()

# Variant 2: SMOTE — subsample to 100k for SMOTE (memory constraint)
# RF is then trained on the resampled data. Evaluation on full test set.
smote_sub = np.random.RandomState(42).choice(len(X_tr_np), 100_000, replace=False)
print(f'  Applying SMOTE to 100k subsample (memory-efficient)...')
smote = SMOTE(random_state=42, k_neighbors=5)
X_res, y_res = smote.fit_resample(X_tr_np[smote_sub], y_train[smote_sub])
print(f'  After SMOTE: {X_res.shape[0]:,} samples  fraud={y_res.mean():.4f}')
rf_sm = RandomForestClassifier(n_estimators=200, max_depth=10,
                                n_jobs=-1, random_state=42)
rf_sm.fit(X_res, y_res)
p_sm  = rf_sm.predict_proba(X_te_np)[:,1]
vp_sm = rf_sm.predict_proba(X_v_np)[:,1]
best_f1, best_thresh = 0, 0.5
for thresh in np.arange(0.01, 0.99, 0.01):
    f1 = f1_fn(y_val, (vp_sm >= thresh).astype(int), zero_division=0)
    if f1 > best_f1: best_f1, best_thresh = f1, thresh
print(f'  [SMOTE] threshold={best_thresh:.2f}  val F1={best_f1:.4f}')
print(f'  Time: {time.time()-t0:.1f}s')
evaluate('RF (SMOTE)', y_test, p_sm, thresh=best_thresh)
del rf_sm, X_res, y_res, p_sm, vp_sm
del X_tr_np, X_te_np, X_v_np; gc.collect()


--- Random Forest (full training set) ---
  Training on 413,614 samples  fraud=0.0350
  [Class Weighted] threshold=0.75  val F1=0.5129
RF (Class Weighted)                 AUC-ROC=0.8854  AUC-PR=0.5178  F1=0.5069  Precision=0.5962  Recall@1%=0.8802  (thresh=0.75)
  Applying SMOTE to 100k subsample (memory-efficient)...
  After SMOTE: 192,846 samples  fraud=0.5000
  [SMOTE] threshold=0.56  val F1=0.4540
  Time: 172.9s
RF (SMOTE)                          AUC-ROC=0.8637  AUC-PR=0.4552  F1=0.4546  Precision=0.5505  Recall@1%=0.8373  (thresh=0.56)


24

In [7]:
import lightgbm as lgb
from sklearn.metrics import f1_score as f1_fn

print('--- LightGBM (scale_pos_weight) ---')
scale_pw = float((y_train==0).sum()) / float((y_train==1).sum())
print(f'  scale_pos_weight = {scale_pw:.1f}')

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.05, num_leaves=127,
    scale_pos_weight=scale_pw, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42
)
t0 = time.time()
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(100)]
)
proba     = lgb_model.predict_proba(X_test)[:,1]
val_proba = lgb_model.predict_proba(X_val)[:,1]
print(f'  Prob range: {proba.min():.4f} – {proba.max():.4f}')
best_f1, best_thresh = 0, 0.5
for thresh in np.arange(0.01, 0.99, 0.01):
    f1 = f1_fn(y_val, (val_proba >= thresh).astype(int), zero_division=0)
    if f1 > best_f1: best_f1, best_thresh = f1, thresh
print(f'  Optimal threshold={best_thresh:.2f}  val F1={best_f1:.4f}')
print(f'  Time: {time.time()-t0:.1f}s')
evaluate('LightGBM', y_test, proba, thresh=best_thresh)
del lgb_model, proba, val_proba; gc.collect()


--- LightGBM (scale_pos_weight) ---
  scale_pos_weight = 27.6
[LightGBM] [Info] Number of positive: 14473, number of negative: 399141
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.721939 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 39111
[LightGBM] [Info] Number of data points in the train set: 413614, number of used features: 431
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034992 -> initscore=-3.317030
[LightGBM] [Info] Start training from score -3.317030
  Prob range: 0.0333 – 0.1305
  Optimal threshold=0.11  val F1=0.4138
  Time: 21.4s
LightGBM                            AUC-ROC=0.8728  AUC-PR=0.3437  F1=0.4180  Precision=0.3754  Recall@1%=0.5977  (thresh=0.11)


168

In [8]:
from sklearn.ensemble import IsolationForest

print('--- Isolation Forest (full training set) ---')
# Unsupervised — SMOTE/class weights do not apply.
# contamination=0.035 sets the expected anomaly proportion to the true fraud rate.
# IsolationForest scales well to large datasets — no subsampling needed.
print(f'  Training on {X_train.shape[0]:,} samples')
t0  = time.time()
iso = IsolationForest(n_estimators=200, contamination=0.035,
                      n_jobs=-1, random_state=42)
iso.fit(X_train.to_numpy(dtype=np.float32))
scores = -iso.score_samples(X_test.to_numpy(dtype=np.float32))
evaluate('Isolation Forest', y_test, scores)
print(f'  Time: {time.time()-t0:.1f}s')
del iso, scores; gc.collect()


--- Isolation Forest (full training set) ---
  Training on 413,614 samples
Isolation Forest                    AUC-ROC=0.7513  AUC-PR=0.0963  F1=0.2018  Precision=0.1324  Recall@1%=0.0610  (thresh=0.50)
  Time: 8.3s


116

In [9]:
print('\n' + '='*75)
print('BASELINE SUMMARY')
print('='*75)
baseline_keys = ['LR (Class Weighted)', 'LR (SMOTE)',
                 'RF (Class Weighted)', 'RF (SMOTE)',
                 'LightGBM', 'Isolation Forest']
baseline_res = {k: results[k] for k in baseline_keys if k in results}
print(pd.DataFrame(baseline_res).T.round(4).to_string())



BASELINE SUMMARY
                     AUC_ROC  AUC_PR      F1  Precision  Recall_at_1pct
LR (Class Weighted)   0.8380  0.2629  0.3304     0.2972          0.5141
LR (SMOTE)            0.8302  0.2457  0.3207     0.2891          0.4847
RF (Class Weighted)   0.8854  0.5178  0.5069     0.5962          0.8802
RF (SMOTE)            0.8637  0.4552  0.4546     0.5505          0.8373
LightGBM              0.8728  0.3437  0.4180     0.3754          0.5977
Isolation Forest      0.7513  0.0963  0.2018     0.1324          0.0610


## Step 5: Graph Construction

We free the tabular splits and build a transaction graph from a **stratified 200k subsample** (all fraud rows + sampled legit rows, preserving the original fraud rate).

**Nodes** = transactions · **Edges** = shared card / email domain / device / address

> *Due to memory constraints inherent to graph construction at scale, GNN experiments were conducted on a stratified subsample of 200k transactions preserving the original class distribution. All models were evaluated on the same held-out test partition.*

In [10]:
# Free tabular splits — no longer needed for graph construction
del X_train, X_val, X_test
gc.collect()

from collections import defaultdict
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, remove_self_loops, coalesce

# ── FULL DATASET — no subsampling ────────────────────────────────────
# All 590k transactions used. All 20,663 fraud cases preserved.
print(f'Building graph on full dataset: {train.shape[0]:,} transactions')
sub = train.reset_index(drop=True)
N   = len(sub)

EDGE_FEATS = ['card1','card2','addr1','addr2',
              'P_emaildomain','R_emaildomain','DeviceInfo']
MAX_GROUP  = 50
edge_list  = []

for feat in EDGE_FEATS:
    if feat not in sub.columns: continue
    vals   = sub[feat].fillna('__NA__').astype(str).values
    groups = defaultdict(list)
    for i, v in enumerate(vals):
        if v not in ('__NA__', '-999', 'nan'): groups[v].append(i)
    batch = []
    for members in groups.values():
        if 2 <= len(members) <= MAX_GROUP:
            m   = np.array(members)
            src = np.repeat(m, len(m))
            dst = np.tile(m, len(m))
            mk  = src != dst
            batch.append(np.stack([src[mk], dst[mk]]))
    if batch:
        ei = np.concatenate(batch, axis=1)
        edge_list.append(ei)
        print(f'  {feat:<20} {ei.shape[1]//2:>8,} edges')

edge_index = torch.tensor(np.concatenate(edge_list, axis=1), dtype=torch.long)
edge_index = to_undirected(edge_index)
edge_index, _ = remove_self_loops(edge_index)
edge_index = coalesce(edge_index, num_nodes=N)
del edge_list; gc.collect()

edge_mem = edge_index.element_size() * edge_index.nelement() / 1e9
print(f'\nNodes     : {N:,}')
print(f'Edges     : {edge_index.shape[1]:,}')
print(f'Avg degree: {edge_index.shape[1]/N:.2f}')
print(f'Edge RAM  : {edge_mem:.2f} GB')
if torch.cuda.is_available():
    free_vram = torch.cuda.mem_get_info()[0]/1e9
    print(f'VRAM free : {free_vram:.1f} GB')
    if edge_mem > free_vram * 0.4:
        print('⚠️  WARNING: Edge index is large relative to available VRAM.')
        print('   If GNNs OOM, revert to 200k subsample.')


Building graph on full dataset: 590,540 transactions
  card1                 912,231 edges
  card2                  10,120 edges
  addr1                   9,039 edges
  addr2                   3,082 edges
  P_emaildomain           3,170 edges
  R_emaildomain           7,942 edges
  DeviceInfo            111,583 edges

Nodes     : 590,540
Edges     : 2,091,012
Avg degree: 3.54
Edge RAM  : 0.03 GB
VRAM free : 15.5 GB


In [11]:
# Node features — numeric columns only, normalised to float32
num_cols = [c for c in sub.select_dtypes(include=np.number).columns
            if c not in ('isFraud','TransactionID')]

X_graph = sub[num_cols].fillna(-999).values.astype(np.float32)
X_graph = StandardScaler().fit_transform(X_graph).astype(np.float32)

node_feats  = torch.tensor(X_graph, dtype=torch.float)
node_labels = torch.tensor(sub['isFraud'].values, dtype=torch.long)
del X_graph; gc.collect()

# Stratified 70 / 15 / 15 split
def split_idx(arr, seed=42):
    arr = np.random.RandomState(seed).permutation(arr)
    n   = len(arr)
    return arr[:int(.70*n)], arr[int(.70*n):int(.85*n)], arr[int(.85*n):]

fi = np.where(sub['isFraud'].values == 1)[0]
li = np.where(sub['isFraud'].values == 0)[0]
tr_f, va_f, te_f = split_idx(fi)
tr_l, va_l, te_l = split_idx(li)

def make_mask(idx_list):
    m = torch.zeros(N, dtype=torch.bool)
    m[np.concatenate(idx_list)] = True
    return m

train_mask = make_mask([tr_f, tr_l])
val_mask   = make_mask([va_f, va_l])
test_mask  = make_mask([te_f, te_l])

data = Data(x=node_feats, edge_index=edge_index, y=node_labels,
            train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)
del sub, node_feats, edge_index; gc.collect()

print(data)
print(f'Train: {train_mask.sum():,}  Val: {val_mask.sum():,}  Test: {test_mask.sum():,}')
if torch.cuda.is_available():
    print(f'VRAM free before training: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')


Data(x=[590540, 401], edge_index=[2, 2091012], y=[590540], train_mask=[590540], val_mask=[590540], test_mask=[590540])
Train: 413,377  Val: 88,581  Test: 88,582
VRAM free before training: 15.5 GB


## Step 6: GNN Model Definitions

In [12]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from torch_geometric.loader import NeighborLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

class GCN(nn.Module):
    def __init__(self, in_ch, hid, out_ch=2, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hid)
        self.conv2 = GCNConv(hid, hid)
        self.lin   = nn.Linear(hid, out_ch)
        self.drop  = dropout
    def forward(self, x, ei):
        x = F.dropout(F.relu(self.conv1(x, ei)), p=self.drop, training=self.training)
        x = F.dropout(F.relu(self.conv2(x, ei)), p=self.drop, training=self.training)
        return self.lin(x)

class GAT(nn.Module):
    def __init__(self, in_ch, hid, out_ch=2, heads=4, dropout=0.3):
        super().__init__()
        self.conv1 = GATConv(in_ch, hid, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hid*heads, hid, heads=1, dropout=dropout)
        self.lin   = nn.Linear(hid, out_ch)
        self.drop  = dropout
    def forward(self, x, ei):
        x = F.dropout(x, p=self.drop, training=self.training)
        x = F.dropout(F.elu(self.conv1(x, ei)), p=self.drop, training=self.training)
        x = F.elu(self.conv2(x, ei))
        return self.lin(x)

class GraphSAGE(nn.Module):
    def __init__(self, in_ch, hid, out_ch=2, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, hid)
        self.conv2 = SAGEConv(hid, hid)
        self.lin   = nn.Linear(hid, out_ch)
        self.drop  = dropout
    def forward(self, x, ei):
        x = F.dropout(F.relu(self.conv1(x, ei)), p=self.drop, training=self.training)
        x = F.dropout(F.relu(self.conv2(x, ei)), p=self.drop, training=self.training)
        return self.lin(x)

class CostSensitiveGraphSAGE(nn.Module):
    """
    GraphSAGE with a learnable 2x2 cost matrix.
    Adapted from: Hu et al. (2024) Cost-Sensitive GNN-Based Imbalanced
    Learning for Mobile Social Network Fraud Detection.
    C[i,j] = learned cost of predicting class j when true class is i.
    """
    def __init__(self, in_ch, hid, out_ch=2, dropout=0.3, fraud_rate=0.035):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, hid)
        self.conv2 = SAGEConv(hid, hid)
        self.lin   = nn.Linear(hid, out_ch)
        self.drop  = dropout
        ir         = fraud_rate / (1.0 - fraud_rate)
        self.C = nn.Parameter(torch.tensor(
            [[0.0, 1.0], [1.0/ir, 0.0]], dtype=torch.float))
    def forward(self, x, ei):
        x = F.dropout(F.relu(self.conv1(x, ei)), p=self.drop, training=self.training)
        x = F.dropout(F.relu(self.conv2(x, ei)), p=self.drop, training=self.training)
        return self.lin(x)
    def cost_sensitive_loss(self, logits, labels):
        probs       = torch.softmax(logits, dim=1)
        C_pos       = torch.clamp(self.C, min=0.0)
        cost_rows   = C_pos[labels]
        sample_cost = (probs * cost_rows).sum(dim=1)
        ce          = F.cross_entropy(logits, labels, reduction='none')
        return (sample_cost * ce).mean()


Training on: cuda


## Step 7: Run GNN Experiments

Mini-batch training using `NeighborLoader` (2-hop, 15+10 neighbours per hop, batch size 2048). This replaces full-batch training and provides:

- **Training stochasticity** — different subgraphs each epoch, acts as regularisation
- **Fraud neighbourhood focus** — fraud nodes sample local context each batch
- **Memory efficiency** — only a subgraph moves to GPU per step

Requires `torch-sparse` which is installed in Step 1.

In [13]:
def run_gnn(model, data, name, epochs=150, lr=1e-3,
            patience=15, cost_sensitive=False,
            batch_size=2048, num_neighbors=[15, 10]):
    """
    Mini-batch GNN training using NeighborLoader.
    Samples a 2-hop subgraph around each seed node per batch,
    providing training stochasticity and better fraud signal isolation.
    """
    model = model.to(device)

    n_pos = int(data.y[data.train_mask].sum())
    n_neg = int(data.train_mask.sum()) - n_pos
    pos_w = torch.tensor([n_neg / n_pos], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    if cost_sensitive:
        gnn_p = [p for n, p in model.named_parameters() if n != 'C']
        opt   = torch.optim.Adam([
            {'params': gnn_p,     'lr': lr},
            {'params': [model.C], 'lr': lr * 0.1}
        ], weight_decay=5e-4)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=0.5, patience=5)

    # ── NeighborLoader setup ──────────────────────────────────────────
    train_loader = NeighborLoader(
        data, num_neighbors=num_neighbors,
        batch_size=batch_size, input_nodes=data.train_mask,
        shuffle=True)
    val_loader = NeighborLoader(
        data, num_neighbors=num_neighbors,
        batch_size=batch_size * 2, input_nodes=data.val_mask,
        shuffle=False)
    test_loader = NeighborLoader(
        data, num_neighbors=num_neighbors,
        batch_size=batch_size * 2, input_nodes=data.test_mask,
        shuffle=False)

    best_auc, best_state, stale = 0, None, 0

    for epoch in range(1, epochs + 1):
        # ── Train ──
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            # Only compute loss on seed nodes (first batch_size nodes)
            logits = model(batch.x, batch.edge_index)[:batch.batch_size]
            labels = batch.y[:batch.batch_size]
            loss = (model.cost_sensitive_loss(logits, labels)
                    if cost_sensitive
                    else criterion(logits[:,1].float(), labels.float()))
            loss.backward()
            opt.step()
            total_loss += loss.item()

        # ── Validate ──
        model.eval()
        vp, vl = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out   = model(batch.x, batch.edge_index)[:batch.batch_size]
                vp.append(torch.softmax(out, dim=1)[:,1].cpu())
                vl.append(batch.y[:batch.batch_size].cpu())
        vp = torch.cat(vp).numpy()
        vl = torch.cat(vl).numpy()
        val_auc = roc_auc_score(vl, vp)
        sched.step(val_auc)

        if epoch % 10 == 0:
            extra = ''
            if cost_sensitive:
                C = model.C.detach().cpu()
                extra = f'  C(FN)={C[1,0]:.3f} C(FP)={C[0,1]:.3f}'
            print(f'  Ep {epoch:3d} | loss={total_loss:.3f} '
                  f'| val AUC={val_auc:.4f}{extra}')

        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            print(f'  Early stop ep {epoch} — best val AUC={best_auc:.4f}')
            break

    # ── Test + threshold optimisation ────────────────────────────────
    model.load_state_dict(best_state)
    model.eval()
    tp, tl = [], []
    vp_final, vl_final = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out   = model(batch.x, batch.edge_index)[:batch.batch_size]
            tp.append(torch.softmax(out, dim=1)[:,1].cpu())
            tl.append(batch.y[:batch.batch_size].cpu())
        for batch in val_loader:
            batch = batch.to(device)
            out   = model(batch.x, batch.edge_index)[:batch.batch_size]
            vp_final.append(torch.softmax(out, dim=1)[:,1].cpu())
            vl_final.append(batch.y[:batch.batch_size].cpu())

    tp = torch.cat(tp).numpy()
    tl = torch.cat(tl).numpy()
    vp_final = torch.cat(vp_final).numpy()
    vl_final = torch.cat(vl_final).numpy()

    from sklearn.metrics import f1_score as f1_fn
    best_f1, best_thresh = 0, 0.5
    for thresh in np.arange(0.01, 0.99, 0.01):
        f1 = f1_fn(vl_final, (vp_final >= thresh).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh

    print(f'  Optimal threshold: {best_thresh:.2f}  (val F1={best_f1:.4f})')
    torch.cuda.empty_cache()
    return tl, tp, best_thresh


In [14]:
IN = data.x.shape[1]
H  = 128
fr = float(data.y[data.train_mask].float().mean())

print(f'Node features : {IN}')
print(f'Hidden dim    : {H}')
print(f'Fraud rate    : {fr:.4f}')
if torch.cuda.is_available():
    print(f'VRAM free     : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

experiments = [
    ('GCN',          GCN(IN, H),                                    False),
    ('GAT',          GAT(IN, H, heads=4),                           False),
    ('GraphSAGE',    GraphSAGE(IN, H),                              False),
    ('CS-GraphSAGE', CostSensitiveGraphSAGE(IN, H, fraud_rate=fr),  True),
]

for name, model, cs in experiments:
    print(f'\n{"+"*55}\n  {name}\n{"+"*55}')
    try:
        t0 = time.time()
        y_true, y_prob, best_thresh = run_gnn(
            model, data, name,
            epochs=150, patience=15,
            cost_sensitive=cs,
            batch_size=2048,
            num_neighbors=[15, 10]
        )
        print(f'  Wall time: {time.time()-t0:.1f}s')
        evaluate(name, y_true, y_prob, thresh=best_thresh)
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM — {name} skipped.')
            torch.cuda.empty_cache()
        else:
            raise
    del model; torch.cuda.empty_cache(); gc.collect()


Node features : 401
Hidden dim    : 128
Fraud rate    : 0.0350
VRAM free     : 15.5 GB

+++++++++++++++++++++++++++++++++++++++++++++++++++++++
  GCN
+++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Ep  10 | loss=196.381 | val AUC=0.8351
  Ep  20 | loss=191.309 | val AUC=0.8434
  Ep  30 | loss=188.880 | val AUC=0.8456
  Ep  40 | loss=181.588 | val AUC=0.8558
  Ep  50 | loss=177.802 | val AUC=0.8606
  Ep  60 | loss=175.802 | val AUC=0.8634
  Ep  70 | loss=173.587 | val AUC=0.8637
  Ep  80 | loss=172.760 | val AUC=0.8648
  Ep  90 | loss=171.882 | val AUC=0.8654
  Ep 100 | loss=171.075 | val AUC=0.8686
  Ep 110 | loss=168.079 | val AUC=0.8705
  Ep 120 | loss=166.400 | val AUC=0.8729
  Ep 130 | loss=164.646 | val AUC=0.8740
  Ep 140 | loss=163.485 | val AUC=0.8740
  Ep 150 | loss=163.514 | val AUC=0.8746
  Optimal threshold: 0.86  (val F1=0.4652)
  Wall time: 4823.4s
GCN                                 AUC-ROC=0.8770  AUC-PR=0.4741  F1=0.4789  Precision=0.5178  Recall@1%=0.8192  (

## Step 8: Final Results

In [15]:
print('\n' + '='*65)
print('FINAL RESULTS — All Models')
print('='*65)
res_df = pd.DataFrame(results).T.sort_values('AUC_ROC', ascending=False)
res_df.index.name = 'Model'
print(res_df.round(4).to_string())

res_df.to_csv('/kaggle/working/experiment_results.csv')
print('\nSaved → /kaggle/working/experiment_results.csv')



FINAL RESULTS — All Models
                     AUC_ROC  AUC_PR      F1  Precision  Recall_at_1pct
Model                                                                  
GraphSAGE             0.8940  0.5231  0.5185     0.6016          0.8689
RF (Class Weighted)   0.8854  0.5178  0.5069     0.5962          0.8802
CS-GraphSAGE          0.8798  0.4496  0.4621     0.5191          0.7627
GCN                   0.8770  0.4741  0.4789     0.5178          0.8192
LightGBM              0.8728  0.3437  0.4180     0.3754          0.5977
RF (SMOTE)            0.8637  0.4552  0.4546     0.5505          0.8373
GAT                   0.8419  0.3278  0.3614     0.3422          0.5887
LR (Class Weighted)   0.8380  0.2629  0.3304     0.2972          0.5141
LR (SMOTE)            0.8302  0.2457  0.3207     0.2891          0.4847
Isolation Forest      0.7513  0.0963  0.2018     0.1324          0.0610

Saved → /kaggle/working/experiment_results.csv
